<h1 style="text-align: center;">Machine Learning Modeling</h1>
<h3 style="text-align: center;">Bank Marketing Campaign</h3>

---

<h5 style="text-align: right;">By Elmar leonard & Nadya Divia Go</h5>

# **Section 1: Model Implementation**

## **1.1 How to Implement the Model?**



* **Scoring Input & Pipeline Automation:**
  * **Required Data:** The serialized pipeline (`tunned_model.pkl`) expects exactly 10 raw CRM fields per customer: `age`, `job`, `balance`, `housing`, `loan`, `contact`, `month`, `campaign`, `pdays`, and `poutcome`.
  * **Zero Preprocessing Overhead:** All specialized feature engineering (such as `age_group`, `pdays_contacted`, and `job_grouped`) executes automatically inside the pipeline via the custom `feature_fixing` transformer. Data engineering pipelines can feed raw data directly into the model.
* **Batch Scoring Workflow:**
  1. **Inference:** A nightly or weekly batch job extracts the current prospect list from the CRM and computes conversion probabilities using `model.predict_proba()`.
  2. **Thresholding:** Applies the optimized classification cutoff of **0.351** to separate customers into "Contact" or "Deprioritize" segments. This threshold can be adjusted dynamically (e.g., raised to 0.75 for strict cost control or lowered to 0.25 for aggressive acquisition) as outlined in Business Understanding Section 5.3.
  3. **Prioritized Routing:** Exports a ranked call sheet ordered by descending probability so sales agents handle high-propensity leads first, maximizing daily conversion velocity.
* **Human-in-the-Loop Integration:**
  * **Decision Support:** The model serves as an intelligent decision-support layer rather than an automated replacement for human teams. It optimizes *who* to call, while sales agents retain full control over the actual client interaction.
* **Continuous Feedback Loop:**
  * **Model Maintenance:** Real-world conversion outcomes (`yes`/`no`) from dialed lists must be captured and appended back to the historical data matrix.
  * **Retraining Cadence:** Scheduled retraining jobs should run periodically (e.g., quarterly) to shield the model from performance degradation caused by macro-economic shifts, interest rate changes, or changing consumer habits.


## **1.2 What are the Limitation of the Model?**

While the final Gradient Boosting pipeline satisfies the primary performance benchmarks, several structural limitations and operational assumptions must accompany its deployment:

* **Operational Dependency vs. Profile Blindness:**
  * **The Issue:** Feature importance and SHAP attributions prove that the model relies almost exclusively on campaign history (`poutcome`, `month`, `contact`) rather than customer profiles (`age`, `job`, `housing`, `loan`). 
  * **Impact:** The model effectively predicts based on *"how well past interactions went"* rather than *"who the consumer is."*
* **Cold-Start Vulnerability:**
  * **The Issue:** Approximately 72% of the dataset consists of fresh prospects with a `poutcome` of `"unknown"`. 
  * **Impact:** For nearly three-quarters of your target lists, the model's strongest predictive signal is entirely absent, forcing it to fall back on weaker administrative indicators. 
  * **Action Required:** An independent performance audit split specifically by `poutcome == "unknown"` vs. known historical segments must be completed before deploying this pipeline universally across fresh marketing leads.
* **Precision Deficit under Optimized Recall:**
  * **The Trade-Off:** To secure the required **≥70% recall target** (achieving 80%), the decision threshold was dialed down to 0.351. This adjustment pulls the test-set precision down to **63%**, which sits marginally below the project's target threshold of ≥65%. 
  * **Adjustment:** The threshold may need to be adjusted back upward if the business priority pivots away from conversion volume and toward call-center staff efficiency.
* **Data and Feature Representation Ceilings:**
  * **The Problem:** Learning curves show that the training and validation gap has flattened out without fully closing. The model has entirely maxed out the predictive value of these 10 baseline columns.
  * **Solution:** Substantial accuracy improvements will not come from collecting more data rows of the same kind; they require new data vectors such as client income, active product holdings, tenure, or digital banking engagement.
* **Exclusion of `duration` (Information Leakage Prevention):**
  * **Design Rationale:** Call duration is a massive predictor but is only captured *after* a phone call takes place. It was correctly omitted from training to prevent temporal leakage in a pre-call targeting tool.
  * **Impact:** While counterfactual sweeps confirm the pipeline successfully ignores this column, this protective constraint intentionally lowers the achievable ROC-AUC ceiling compared to standard benchmarks that include talk time.
* **Unvalidated Economic and Environmental Assumptions:**
  * **Financial Metrics:** The business case valuations rely entirely on illustrative operational costs and deposit conversion rewards. These must be replaced with the bank's true unit economics before locking in budgeting decisions.
  * **Macro Drift:** The historical data captures a specific macroeconomic snapshot and seasonal landscape. Shifts in current market interest rates or changing consumer behaviors will degrade predictions over time, mandating a strict quarterly retraining schedule.


## **1.3 Business Calculation?**

In [1]:
TN, FP, FN, TP = 495, 340, 142, 584
n_test = TN + FP + FN + TP
actual_positive = TP + FN
base_rate = actual_positive / n_test

cost_per_call = 5        
value_per_deposit = 92

# Strategy A: Blanket "contact-all" campaign (what the source data actually reflects)
blanket_calls = n_test
blanket_conversions = actual_positive
blanket_cost = blanket_calls * cost_per_call
blanket_revenue = blanket_conversions * value_per_deposit
blanket_net = blanket_revenue - blanket_cost

# Strategy B: Model-guided targeting (call only predicted positives)
model_calls = TP + FP
model_conversions = TP
model_precision = model_conversions / model_calls
model_cost = model_calls * cost_per_call
model_revenue = model_conversions * value_per_deposit
model_net = model_revenue - model_cost

# Strategy C: Random dialing of the SAME call volume as the model (fair efficiency comparison)
random_calls = model_calls
random_conversions = random_calls * base_rate
random_revenue = random_conversions * value_per_deposit
random_cost = random_calls * cost_per_call
random_net = random_revenue - random_cost

calls_saved = blanket_calls - model_calls
cost_saved = calls_saved * cost_per_call
opportunity_cost = FN * value_per_deposit
uplift_pct = (model_precision / base_rate - 1) * 100

print(f"Test set size: {n_test} | Actual subscribers (base rate): {actual_positive} ({base_rate:.1%})")
print(f"Assumptions -> cost/call: ${cost_per_call}, value/deposit: ${value_per_deposit}\n")

print("=== Strategy A: Blanket contact-all campaign ===")
print(f"Calls: {blanket_calls} | Conversions: {blanket_conversions} | Cost: ${blanket_cost:,} | Revenue: ${blanket_revenue:,} | Net: ${blanket_net:,}\n")

print("=== Strategy B: Model-guided targeting (threshold = 0.351) ===")
print(f"Calls: {model_calls} | Conversions: {model_conversions} | Precision: {model_precision:.1%}")
print(f"Cost: ${model_cost:,} | Revenue: ${model_revenue:,} | Net: ${model_net:,}\n")

print("=== Strategy C: Random dialing, same volume as Strategy B ===")
print(f"Calls: {random_calls} | Expected conversions: {random_conversions:.1f} | Cost: ${random_cost:,.0f} | Revenue: ${random_revenue:,.0f} | Net: ${random_net:,.0f}\n")

print("=== Model value vs. Random targeting (equal cost, equal call volume) ===")
print(f"Extra conversions captured: +{model_conversions-random_conversions:.1f}  ({uplift_pct:.1f}% relative lift over base rate)")
print(f"Extra revenue at equal cost: ${model_revenue-random_revenue:,.0f}\n")

print("=== Model value vs. Blanket campaign ===")
print(f"Calls avoided: {calls_saved} (-{calls_saved/blanket_calls:.1%})  ->  Operational cost saved: ${cost_saved:,}")
print(f"Subscribers missed (False Negatives): {FN}  ->  Opportunity cost: ${opportunity_cost:,}")

Test set size: 1561 | Actual subscribers (base rate): 726 (46.5%)
Assumptions -> cost/call: $5, value/deposit: $92

=== Strategy A: Blanket contact-all campaign ===
Calls: 1561 | Conversions: 726 | Cost: $7,805 | Revenue: $66,792 | Net: $58,987

=== Strategy B: Model-guided targeting (threshold = 0.351) ===
Calls: 924 | Conversions: 584 | Precision: 63.2%
Cost: $4,620 | Revenue: $53,728 | Net: $49,108

=== Strategy C: Random dialing, same volume as Strategy B ===
Calls: 924 | Expected conversions: 429.7 | Cost: $4,620 | Revenue: $39,536 | Net: $34,916

=== Model value vs. Random targeting (equal cost, equal call volume) ===
Extra conversions captured: +154.3  (35.9% relative lift over base rate)
Extra revenue at equal cost: $14,192

=== Model value vs. Blanket campaign ===
Calls avoided: 637 (-40.8%)  ->  Operational cost saved: $3,185
Subscribers missed (False Negatives): 142  ->  Opportunity cost: $13,064


This section translates the model's confusion matrix into concrete business metrics, comparing targeting strategies under illustrative financial assumptions:

* **Model vs. Random Selection (Fixed Call Volume Baseline):**
  * **The Comparison:** Isolates the exact value of *predictive targeting* by holding total call capacity equal at 924 contacts.
  * **The Return:** The model captures **~154 additional subscribers** compared to a random dialing approach.
  * **Efficiency Lift:** This represents a **35.9% relative lift in conversion rate** (moving from a 46.5% base rate up to a 63.2% model precision), offering a defensible efficiency gain for a fixed operational budget.
* **Model vs. Blanket Campaign ("Call Everyone"):**
  * **The Trade-Off:** Highlights the direct financial tension between precision and recall.
  * **Cost Saved:** The model saves **~$3,185** in call center labor and telecom costs by successfully filtering out 637 low-propensity leads.
  * **Opportunity Cost:** The model misses 142 true subscribers (False Negatives), representing **~$28,400** in uncaptured deposit revenue.
  * **Volume Constraint:** Blanket calling yields higher raw revenue in this specific backtest because the pool is small enough to reach everyone. However, it requires double the operational budget and does not scale as the list grows.
* **Core Strategic Takeaways:**
  * **Capacity Optimization:** The true business value of this pipeline is optimizing a **finite sales team capacity**. In the real world, staff hours are capped; this model ensures those hours are focused entirely on the highest-propensity targets.
  * **Operational Pilot Requirement:** These financial outcomes are entirely **illustrative**. The bank must plug in its actual unit economics (exact cost-per-call and true lifetime value of a deposit) and run a live pilot campaign to validate these precision numbers before committing major marketing budgets.


# **Section 2: Conclusion & Recommendation**

## **2.1 Conclusion**

### **2.1.1 Model**

* **Champion Architecture Selection:**
  * **The Model:** `Gradient Boosting` was crowned the champion architecture after a broad benchmarking phase across linear and tree-based model families.
  * **The Search:** A localized randomized search locked in a Cross-Validation ROC-AUC of **0.7762**. A subsequent exhaustive grid search confirmed this was the empirical performance plateau, returning a statistically negligible **+0.0017** gain.
* **Final Test Set Performance Summary (Threshold = 0.351):**
  * **Discrimination:** `ROC-AUC` = **79.6%**
  * **Class `yes` Metrics:** `Precision` = **63%** | `Recall` = **80%** | `F1-Score` = **0.71**
  * **Confusion Matrix Breakdown (n = 1,561):** 
    * *True Positives (TP):* 584 | *False Positives (FP):* 340 
    * *False Negatives (FN):* 142 | *True Negatives (TN):* 495
* **Scorecard Against Business Success Criteria:**

| Criterion | Target | Achieved | Status |
| :--- | :---: | :---: | :--- |
| **ROC-AUC** | ≥ 0.75 | **0.796** | Passed (Strong global discrimination) |
| **F1-Score (`yes`)** | ≥ 0.65 | **0.71** | Passed (Optimized balance) |
| **Recall (`yes`)** | ≥ 70% | **80%** | Passed (Exceeds volume targeting goals) |
| **Precision (`yes`)** | ≥ 65% | **63%** | Marginally short (Can be resolved via threshold adjustment) |
| **Generalization Gap** | ≤ 5% | **~2–3%** | Passed (Zero evidence of quiet overfitting) |

* **Reliability and Deployment Readiness:**
  * **Inherent Calibration:** The model is exceptionally well-calibrated out of the box. Wrapping the pipeline in `CalibratedClassifierCV` yielded a completely flat Brier score change (0.1796 → 0.1795). Raw `predict_proba()` values can be trusted immediately for tactical financial routing.
  * **Triangulated Explainability:** Impurity-based weights, permutation importance, and game-theoretic SHAP values all converge on an identical story: `poutcome`, `month`, and `contact` dictate the model's choices. This multi-method alignment satisfies the "no black box" compliance mandate.
* **The Bottom Line:** The model satisfies 4 of the 5 hard data-science success criteria outright, with the remaining precision metric missing by a negligible margin due to an aggressive recall trade-off. This represents a genuinely valuable, deployable asset ready to drive marketing efficiency.


### **2.1.1 Business**

* **Direct Answers to Core Business Questions:**
  * **The Ideal Profile:** Demographics matter significantly less than expected. Features like `age`, `job`, `housing`, and `loan` carry near-zero model weight. Predictive success is dictated by **operational history and logistics**, not who the consumer is on paper.
  * **The Core Drivers:** A customer's prior campaign outcome (`poutcome`) dominates model decisions by a wide margin, followed by the contact `month` and interaction `contact` channel. This hierarchy was successfully cross-validated across three independent vectors: EDA Cramér's V rankings (0.30 correlation boundaries), univariate feature selection, and game-theoretic SHAP values.
  * **Operational Tactics:** Timing and channel quality alter conversion rates. Off-peak outreach months (`mar`, `sep`, `oct`, `dec`) generate **3x to 8x higher conversion rates** than high-volume peak periods. Furthermore, poorly logged contact methods (`unknown`) convert at roughly half the rate of verified `cellular` or `telephone` interactions.
  * **High-Yield Targeting:** Students, retirees, debt-free consumers, and individuals with a prior `success` history represent your most lucrative sub-segments, posting **67% to 92% conversion rates** against the baseline 46–48% average.
  * **Bottom-Line Impact:** Holding total outreach capacity equal, model-driven lead routing captures a **~36% relative increase in successful conversions** compared to a standard random dialing script.

* **Resolving Operational Challenges:**
  * **Targeted Allocation:** The model solves the problem of *whom* to call by ranking leads by conversion propensity and filtering out cold, low-value records. Simultaneously, the exploratory analysis provides a playbook for *when* and *how* to approach those targets to optimize resource consumption.

* **The "May" Scheduling Anomaly (High-Leverage Pivot):**
  * **The Finding:** The single most impactful business discovery requires no machine learning code to exploit. The bank's highest-volume outreach month (`may`) is simultaneously its **lowest-converting month by a wide margin**, independent of customer traits. 
  * **Strategic Recommendation:** Rebalancing call-center schedules away from this saturated period and smoothing the pipeline into higher-yield off-peak months represents an immediate, zero-deployment victory that will lift revenue out of the gate.

* **Data Governance and CRM Quality Gaps:**
  * **The Finding:** The large volume of records categorized as an `unknown` contact type correlates heavily with bottom-tier conversion performance. This pattern indicates an underlying tracking, attribution, or data-logging failure within the frontline CRM platform rather than a genuine behavioral trait of the customer. 
  * **Strategic Recommendation:** This attribution gap should be flagged for the data engineering and systems administration teams to enforce stricter call-logging compliance, cleaning up downstream data quality for future modeling runs.


## **2.1 Recommendation**

### **2.2.1 Model**

To successfully operationalize this machine learning pipeline and maximize campaign returns, the following deployment roadmap is recommended:

* **Deploy as a Decision-Support Scoring Layer:**
  * Integrate the tuned Gradient Boosting pipeline directly with the CRM to feed the sales team a probability-ranked call sheet rather than a restrictive binary cutoff. This preserves human agent judgment while ensuring daily dialing hours are focused entirely on the highest-propensity leads.
* **Maintain an Adjustable Operational Threshold:**
  * Avoid fixing the classification cutoff at a permanent **0.351**. Treat the threshold as a configurable business lever aligned with the strategy in Business Understanding Section 5.3. 
  * *Tactical Rule:* Raise the threshold (e.g., toward 0.75) during tight operational constraints to maximize precision and call efficiency, or lower it (e.g., toward 0.25) to prioritize aggressive market-share growth. Because the current precision (63%) sits just below the initial 65% target, teams prioritizing staff efficiency should default slightly higher than 0.351.
* **Segment Evaluation and Auditing by Lead History:**
  * Run separate ROC-AUC, precision, and recall evaluations for the `poutcome == "unknown"` segment (cold, net-new prospects) vs. known historical clients. Because the model's dominant predictive signal disappears for the cold segment, separate performance benchmarking is required before trusting the pipeline equally across mixed lists.
* **Establish a Scheduled Retraining Cadence:**
  * Re-fit the model on a standard quarterly cycle. Create an automated data pipeline to feed actual conversion outcomes back into the historical training matrix, shielding the model from performance degradation caused by shifting macroeconomic climates, rate changes, or changing consumer habits.
* **Pivot Focus from Data Volume to Feature Enrichment:**
  * Stop investing resources into collecting more rows of the same data. Learning curves prove the current pipeline has hit an informational ceiling. 
  * To break through the current **~0.78 ROC-AUC ceiling**, invest data engineering effort into capturing entirely new feature dimensions, such as consumer digital engagement metrics, income brackets, product tenure, or multi-channel response history.
* **Execute a Controlled Live Pilot (A/B Test):**
  * Before rolling out full deployment, run a live pilot campaign on a held-out slice of prospects. Benchmark model-guided call sheets directly against the legacy blanket outreach method to substitute the project's illustrative cost-per-call and deposit-value metrics with true, verified unit economics.


### **2.2.2 Business**

The final phase of this roadmap outlines high-leverage, operational adjustments designed to maximize outreach efficiency and lift sales velocity without requiring additional marketing spend:

* **De-Saturate and Rebalance the Campaign Calendar:**
  * **The Problem:** May is simultaneously the bank's highest-volume and lowest-converting month (**33.4% conversion** vs. the 47.8% overall baseline).
  * **The Fix:** Shift operational call volume away from May and redistribute it toward highly lucrative off-peak windows like `mar`, `sep`, `oct`, and `dec` (**85–89% conversion**). This simple calendar rebalancing directly boosts ROI without requiring fresh leads.
* **Codify High-Yield Segments into Sales Playbooks:**
  * Explicitly prioritize top-tier cohorts in daily targeting rules and tailor specialized communication scripts for them. These core profiles include: students (**74.5% conversion**), retirees (**67.6% conversion**), debt-free prospects (no housing or personal loans), and clients with a prior `poutcome == "success"` (**91.5% conversion**).
* **Launch a Structured "Second-Touch" Campaign for Past Decliners:**
  * **The Insight:** Do not discard prospects who rejected previous campaigns. Customers with a prior `poutcome == "failure"` still convert at a remarkable **51.3% rate**, outperforming entirely cold, uncontacted leads (**40.8%**). 
  * **The Strategy:** Re-routing sales resources to build a structured follow-up track for past decliners is mathematically more efficient than exhausting staff on completely raw prospects.
* **Remediate the `unknown` Contact Logging Gap:**
  * Because contacts tracked as an `unknown` communication channel convert at roughly half the rate of verified `cellular` or `telephone` outreach, an immediate operational audit is required. Determine if this reflects an unoptimized channel or simply incomplete CRM logging by agents. Treat this as an actionable target exclusion rule or an urgent internal data-quality correction.
* **Enforce Strict Caps on Contact Attempts (2–3 Calls Max):**
  * **The Backtest Evidence:** EDA establishes that conversions drop off sharply after a few contacts, and local counterfactual analysis proves that increasing attempts from 1 to 8 slices success probabilities from **0.367 down to 0.181**.
  * **The Rule:** Enforce an operational ceiling of **2 to 3 contact attempts per customer**. Repeated dialing past this threshold yields diminishing returns, inflates labor costs, and actively damages customer sentiment.
* **Match Call-List Boundaries to Real Workforce Capacity:**
  * Instead of attempting to exhaustively dial the entire prospect database, use the model's probability scores to dynamically size weekly call lists to match the exact bandwidth of the sales team. As demonstrated in the financial backtest (Section 1.3), the most defensible commercial value of predictive modeling is maximizing the conversion output extracted from a **fixed operational budget**.
